In [ ]:
# remove unnecessary image from pdr-test lit for download of RDN HDRs

import pandas as pd
import ast

df = pd.read_csv("/home/bekah/pdr-tests/pdr_tests/definitions/ch1/M3_L1B_RDN_HDR_olf.csv")

def extract_rdn_hdr(s):
    s = s.replace('""', '"')
    lst = ast.literal_eval(s)
    
    lst = [f for f in lst if f.endswith("RDN.HDR")]
    
    return '["' + '", '.join(lst) + '"]'

df["files"] = df["files"].apply(extract_rdn_hdr)

df.to_csv("M3_L1B_RDN_HDR.csv", index=False)

In [ ]:
# let's pull info from each HDR

from pathlib import Path
import pandas as pd
import re

folder = Path("/home/bekah/m3_papers/ch1/M3_L1B_RDN_HDR")

rows = []

for hdr in folder.glob("*.HDR"):
    
    text = hdr.read_text()

    name = hdr.stem  
    parts = name.split("_")
    
    ID = parts[0]
    VERSION = parts[1]
    TYPE = parts[0][2]

    def grab(pattern):
        m = re.search(pattern, text)
        return m.group(1).strip() if m else None

    # things we want 
    temperature = grab(r"temperature\s*=\s*([0-9.+-]+)")
    beta_angle = grab(r"beta angle\s*=\s*([0-9.+-]+)")
    dark_signal = grab(r"dark signal image\s*=\s*(\S+)")
    bad_detector = grab(r"bad detector element map\s*=\s*(\S+)")
    flat_field = grab(r"flat field image\s*=\s*(\S+)")
    cal_version = grab(r"Calibrated Data version\s*([^\;]+)")

    rows.append({
        "obs_filename": hdr.name,
        "id": ID,
        "obs_type": TYPE,
        "version": VERSION,
        "obs_temperature": float(temperature) if temperature else None,
        "obs_beta_angle": float(beta_angle) if beta_angle else None,
        "cal_dark_signal_image": dark_signal,
        "cal_bad_detector_map": bad_detector,
        "cal_flat_field_image": flat_field,
        "calibrated_data_version": cal_version
    })

df = pd.DataFrame(rows)

df

In [ ]:
import matplotlib.pyplot as plt
df['obs_type_bin'] = df['obs_type'] == 'T'
plt.scatter(df['obs_temperature'], df['obs_beta_angle'],s=2, c=df['obs_type_bin'])

In [ ]:
df.to_csv("l1b_rdn_hdr_cal_info.csv")

In [ ]:
len(df['id'].unique())

In [ ]:
df['obs_type'].unique()

In [ ]:
df['calibrated_data_version'].unique()

In [ ]:
len(df['cal_flat_field_image'].unique())

In [ ]:
len(df['cal_dark_signal_image'].unique())

In [ ]:
len(df['cal_bad_detector_map'].unique())

In [ ]:
inventory = pd.read_csv('collection_data_inventory.csv',delimiter=':')

In [ ]:
inventory

In [ ]:
parts = inventory["data_readme"].str.split("_", expand=True)
df_l1b = inventory[parts[1] == "l1b"]

In [ ]:
df_l1b

In [ ]:
names = df_l1b["data_readme"].str.split("_", expand=True)

In [ ]:
names[0] = names[0].str.upper()

In [ ]:
names

In [ ]:
merged = names.merge(df, left_on=0, right_on="id", how="outer", indicator=True)

In [ ]:
merged

In [ ]:
df1_only = merged[merged["_merge"] == "left_only"]


In [ ]:
df1_only

In [ ]:
import pyarrow.parquet as pq 

man = pq.read_table("/home/bekah/pdr-tests/pdr_tests/node_manifests/chan1_m3_jpl_bekah.parquet").to_pandas()

In [ ]:
man[f

In [ ]:
man = man[man["filename"].str.contains("_RDN.HDR", regex=False)]


In [ ]:
strings_to_match = df1_only[0].tolist()

pattern = "|".join(strings_to_match)

df_matched = man[man["filename"].str.contains(pattern, regex=True)]

df_matched = df_matched.reset_index(drop=True)

In [ ]:
df_matched

In [ ]:
# ADD MISSING FILES TO SHEET 

In [ ]:
import pandas as pd


base_url = "http://planetarydata.jpl.nasa.gov/"

df_formatted = pd.DataFrame()

df_formatted["label_file"] = df_matched["filename"].str.replace("_RDN.HDR", "_L1B.LBL", regex=False)

df_formatted["files"] = df_matched["filename"].apply(lambda x: f'["{x}"]')

df_formatted["product_id"] = df_matched["filename"].str.replace(".HDR", "", regex=False)

df_formatted["url_stem"] = df_matched["url"].apply(lambda x: base_url + x.split("/...")[0])

df_formatted = df_formatted[["label_file", "files", "product_id", "url_stem"]]

df_formatted.to_csv("RDN_HDR_EXTRAS.csv", index=False)

In [ ]:
import pandas as pd 

df = pd.read_csv("l1b_rdn_hdr_cal_info.csv")

In [ ]:
df

In [ ]:
df['bde'] = df['cal_bad_detector_map'].str.split("_", expand=True)[0]
df['dark'] = df['cal_dark_signal_image'].str.split("_", expand=True)[0]
df['flat'] = df['cal_flat_field_image'].str.split("_", expand=True)[0]


In [ ]:
df[df['flat'] != df['id']]

In [ ]:
df[df['dark'] != df['bde']]